# **2. Генерация признаков**

После предварительного анализа переходим уже к построению самих признаков для обучения модели.

**Цель:** превратить сырые события в таблицу признаков (одна строка = одна cookie_id).

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path("../data")

## **2.1. Загрузка данных**

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
for col in ["cookie_created_at", "window_start_ts", "window_end_ts"]:
    train[col] = pd.to_datetime(train[col])

test = pd.read_csv(DATA_DIR / "test.csv")
for col in ["cookie_created_at", "window_start_ts", "window_end_ts"]:
    test[col] = pd.to_datetime(test[col])

events = pd.read_csv(DATA_DIR / "events.csv")
events["event_ts"] = pd.to_datetime(events["event_ts"])

print(f"\nTrain: {train.shape}")
print(f"Test: {test.shape}")
print(f"Events: {events.shape}")


Train: (11091, 5)
Test: (4909, 4)
Events: (328905, 14)


Получим объединенную таблицу трейна и событий с фильтрацией по временному окну.

In [4]:
def filter_events_by_window(events_df, targets_df):
    """
    Фильтрует события, оставляя только те, что попали в окно наблюдения.
    Возвращает events + target (если есть).
    """
    # Берем нужные колонки из targets
    cols = ["cookie_id", "window_start_ts", "window_end_ts"]
    if "target" in targets_df.columns:
        cols.append("target")
    
    # Merge и фильтрация
    merged = events_df.merge(targets_df[cols], on="cookie_id", how="inner")
    mask = (
        (merged["event_ts"] >= merged["window_start_ts"]) & 
        (merged["event_ts"] <= merged["window_end_ts"])
    )
    filtered = merged.loc[mask].copy()
    
    # Удаляем служебные колонки окна
    filtered = filtered.drop(columns=["window_start_ts", "window_end_ts"])
    return filtered

In [5]:
events_train = filter_events_by_window(events, train)
print(f"Событий в окне (train): {len(events_train)}")

events_test = filter_events_by_window(events, test)
print(f"Событий в окне (test): {len(events_test)}")

Событий в окне (train): 198504
Событий в окне (test): 89690


## **2.2. Генерация признаков**

Собираем все инсайты из EDA в один набор данных.

### **2.2.1. Базовые счетчики**

Гипотезы:

* Боты делают больше событий за 24 часа (парсят активно)

* Боты смотрят больше уникальных товаров (unique_items выше)

* Боты фокусируются на узкой нише (unique_categories ниже)

In [59]:
base_features = events_train.groupby("cookie_id").agg(
    event_count=("event_name", "size"),             # общее число событий
    unique_items=("item_id", "nunique"),            # уникальных товаров
    unique_categories=("item_category", "nunique"), # уникальных категорий
    unique_locations=("item_location", "nunique"),  # уникальных локаций
    unique_event_types=("event_name", "nunique"),   # уникальных типов событий
).reset_index()

# Объединяем с target для train
base_features = base_features.merge(
    train[["cookie_id", "target"]], on="cookie_id", how="left"
)

print(f"Shape: {base_features.shape}")
display(base_features.head())

Shape: (11091, 7)


,cookie_id,event_count,unique_items,unique_categories,unique_locations,unique_event_types,target
0,ck_000c95f1408dcb00,16,9,2,8,4,0
1,ck_000e8c52636e3bec,34,20,4,12,7,0
2,ck_0010e31baa4a1fb7,16,8,3,4,6,0
3,ck_0010ec3874fb5378,74,31,2,19,6,0
4,ck_001722063b94cae0,15,8,3,9,4,0


### **2.2.2. user_agent признаки**

Гипотеза: боты часто используют headless-браузеры, скрипты на Python/curl, имеют короткие или стандартные User-Agent строки. Люди используют обычные браузеры с длинными UA.

Признаки:

- `ua_length` - медианная длина строки User-Agent (короткая = бот)

- `has_headless` - флаг HeadlessChrome

- `has_bot_pattern` - флаг bot-паттернов

- `n_unique_ua` - количество уникальных UA

In [60]:
import re

# Предобработка
ua = events_train[['cookie_id', 'user_agent']].copy()
ua['ua_clean'] = ua['user_agent'].fillna('').str.lower()
ua['ua_len'] = ua['ua_clean'].str.len()

# флаг наличия бот-паттернов и флаг headless
BOT_PATTERNS = re.compile(r'bot|crawler|spider|python|curl|wget|scrapy|selenium')
ua['is_headless'] = ua['ua_clean'].str.contains('headless', regex=False)
ua['is_bot'] = ua['ua_clean'].str.contains(BOT_PATTERNS)

# Аагрегация
ua_features = ua.groupby('cookie_id').agg(
    ua_length=('ua_len', 'median'),
    has_headless=('is_headless', 'any'),
    has_bot_pattern=('is_bot', 'any'),
    n_unique_ua=('user_agent', 'nunique'),
).reset_index()

ua_features[['has_headless', 'has_bot_pattern']] = \
    ua_features[['has_headless', 'has_bot_pattern']].astype(int)

base_features = base_features.merge(ua_features, on='cookie_id', how='left')

### **2.2.3. Признаки платформы**

Гипотеза: боты доминируют на Web-платформе, тогда как люди более равномерно распределены между мобильными и веб-платформами.

In [61]:
# Предобработка: приводим к единому формату (как в EDA)
events_train['platform_clean'] = events_train['platform'].str.lower()
events_train['platform_clean'] = events_train['platform_clean'].replace({'desktop': 'web', 'iphone': 'ios'})

# Считаем долю каждой платформы для каждой куки
platform_counts = (
    events_train.groupby('cookie_id')['platform_clean']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reset_index()
)

platform_counts = platform_counts.rename(columns={
    'web': 'platform_web_ratio',
    'android': 'platform_android_ratio',
    'ios': 'platform_ios_ratio'
})

# Если какой-то платформы не было у куки — заполняем нулем
for col in ['platform_web_ratio', 'platform_android_ratio', 'platform_ios_ratio']:
    if col not in platform_counts.columns:
        platform_counts[col] = 0.0

# Объединяем с предыдущими признаками
base_features = base_features.merge(
    platform_counts[['cookie_id', 'platform_web_ratio', 'platform_android_ratio', 'platform_ios_ratio']],
    on='cookie_id', how='left'
)

**Что получили:**

- `platform_web_ratio` - доля событий с Web

- `platform_android_ratio` - доля событий с Android

- `platform_ios_ratio` - доля событий с iOS

### **2.2.4. Признаки из seller_type**

Гипотеза: боты могут фокусироваться на определенных типах продавцов (например, только профессионалы), тогда как люди более разнообразны в выборе. Также отсутствие информации о продавце (unknown) может указывать на события поиска, где боты активнее.

In [62]:
# Заполняем пропуски значением 'unknown'
events_train['seller_type_clean'] = events_train['seller_type'].fillna('unknown')

# Считаем долю каждого типа продавца для каждой куки
seller_counts = (
    events_train.groupby('cookie_id')['seller_type_clean']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reset_index()
)

# Переименовываем колонки
seller_counts = seller_counts.rename(columns={
    'private': 'seller_private_ratio',
    'pro': 'seller_pro_ratio',
    'unknown': 'seller_unknown_ratio'
})

# Если какого-то типа не было у куки — заполняем нулем
for col in ['seller_private_ratio', 'seller_pro_ratio', 'seller_unknown_ratio']:
    if col not in seller_counts.columns:
        seller_counts[col] = 0.0

# Объединяем с предыдущими признаками
base_features = base_features.merge(
    seller_counts[['cookie_id', 'seller_private_ratio', 'seller_pro_ratio', 'seller_unknown_ratio']],
    on='cookie_id', how='left'
)

**Что получили:**

- `seller_private_ratio` - доля событий с частными продавцами

- `seller_pro_ratio` - доля событий с профессиональными продавцами

- `seller_unknown_ratio` - доля событий без информации о продавце

### **2.2.5. Временные признаки**

Гипотезы:

- Возраст куки: Боты часто создают "свежие" куки специально для парсинга. Люди обычно имеют куки, которые существуют месяцами или годами.

- Ночная активность: Люди спят ночью, боты работают 24/7.

- Скорость действий: Боты делают события быстрее (короткие интервалы).

Признаки возраста куки:

- `is_created_in_window` - флаг: кука создана внутри окна

- `cookie_age_log` - логарифм возраста (для сглаживания выбросов)

In [63]:
# Возраст куки в днях на момент начала окна наблюдения
train['cookie_age_days'] = (
    (train['window_start_ts'] - train['cookie_created_at']).dt.total_seconds() / 86400
)

# Логарифм возраста куки
train['cookie_age_log'] = np.log1p(train['cookie_age_days'].clip(lower=0))

# Флаг: кука создана внутри окна наблюдения
train['is_created_in_window'] = (
    train['cookie_created_at'] >= train['window_start_ts']
).astype(int)

Временные паттерны:

- `night_activity_ratio` - доля событий ночью 0-5 часов

- `day_activity_ratio` - доля событий днем 6-23 часа

- `hour_std` - стандартное отклонение по часам

In [64]:
events_train['hour'] = events_train['event_ts'].dt.hour
time_features = events_train.groupby('cookie_id').agg(
    night_activity_ratio=('hour', lambda x: ((x >= 0) & (x < 6)).mean()),
    day_activity_ratio=('hour', lambda x: ((x >= 6) & (x < 24)).mean()),
    hour_std=('hour', 'std'),
).reset_index()

Интервалы между событиями:

- `median_interval` - медианный интервал между событиями

- `mean_interval` - средний интервал

- `min_interval` - минимальный интервал

- `ratio_instant_01` - доля событий с интервалом < 0.1 сек 

In [65]:
events_sorted = events_train.sort_values(['cookie_id', 'event_ts'])

# Считаем разницу между текущим и предыдущим событием для каждой куки
events_sorted['time_diff_sec'] = events_sorted.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()

# Агрегируем интервалы по кукам
interval_features = events_sorted.groupby('cookie_id').agg(
    median_interval=('time_diff_sec', 'median'),
    mean_interval=('time_diff_sec', 'mean'),
    min_interval=('time_diff_sec', 'min'),
    ratio_instant_01=('time_diff_sec', lambda x: (x < 0.1).mean()),
).reset_index()

Теперь объединим все временные фичи:

In [66]:
age_features = train[['cookie_id', 'cookie_age_log', 'is_created_in_window']].copy()

# Объединяем все временные признаки
time_all = age_features.merge(time_features, on='cookie_id', how='left')
time_all = time_all.merge(interval_features, on='cookie_id', how='left')

# Добавляем к основным признакам
base_features = base_features.merge(time_all, on='cookie_id', how='left')

# Заполняем возможные пропуски (например, если у куки только 1 событие — нет интервалов)
base_features = base_features.fillna({
    'hour_std': 0,
    'median_interval': 0,
    'mean_interval': 0,
    'min_interval': 0,
    'ratio_instant_01': 0
})

### **2.2.6. Поисковые признаки**

Признаки:

- `search_query_ratio` - доля событий с поисковым запросом

- `unique_search_queries` - количество уникальных поисковых запросов

- `max_search_page` - максимальная глубина страницы выдачи

- `mean_search_page` - средняя глубина страницы выдачи

In [67]:
search_events = events_train[events_train['search_query'].notna()]

search_all = (
    events_train.groupby('cookie_id')
    .size().reset_index(name='total_events')
    .merge(
        search_events.groupby('cookie_id').agg(
            search_events_count=('search_query', 'size'),
            unique_search_queries=('search_query', 'nunique'),
            max_search_page=('search_page', 'max'),
            mean_search_page=('search_page', 'mean'),
        ).reset_index(),
        on='cookie_id', how='left',
    )
)
search_all[['search_events_count', 'unique_search_queries',
            'max_search_page', 'mean_search_page']] = \
    search_all[['search_events_count', 'unique_search_queries',
                'max_search_page', 'mean_search_page']].fillna(0)
search_all['search_query_ratio'] = search_all['search_events_count'] / search_all['total_events']

base_features = base_features.merge(
    search_all[['cookie_id', 'search_query_ratio', 'unique_search_queries',
                'max_search_page', 'mean_search_page']],
    on='cookie_id', how='left',
)

### **2.2.7. Признаки координат курсора**

- `pointer_ratio` - доля событий с координатами.

- `has_any_pointer` - флаг наличия данных курсора.

- `pointer_x_range`, `pointer_y_range` - диапазон движения курсора.

In [68]:
events_train['has_pointer'] = events_train['pointer_x'].notna() & events_train['pointer_y'].notna()

pointer_features = events_train.groupby('cookie_id').agg(
    pointer_ratio=('has_pointer', 'mean'),
    has_any_pointer=('has_pointer', 'any'),
    pointer_x_range=('pointer_x', np.ptp),
    pointer_y_range=('pointer_y', np.ptp),
).reset_index()

base_features = base_features.merge(pointer_features, on='cookie_id', how='left')

base_features[['pointer_ratio', 'pointer_x_range', 'pointer_y_range']] = \
    base_features[['pointer_ratio', 'pointer_x_range', 'pointer_y_range']].fillna(0.0)
base_features['has_any_pointer'] = base_features['has_any_pointer'].fillna(False).astype(int)

### **2.2.8. Признаки типов событий**

Доли типов событий:

- `ratio_item_view`, `ratio_photo_swipe`, `ratio_favorite_add`, `ratio_login`, `ratio_search_results_view`, `ratio_contact_phone_show`, `ratio_contact_chat_open`, `ratio_contact_message_sent`, `ratio_seller_page_view`, `ratio_captcha_shown`

Бинарные флаги:

- `has_favorite_add` - добавлял ли в избранное

- `has_login` - логинился ли

- `has_captcha` - была ли показана капча

Отношения:

- `item_view_to_photo_ratio` - отношение просмотров к листанию фото

In [69]:
# Доли типов событий для каждой куки
event_type_counts = (
    events_train.groupby('cookie_id')['event_name']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .add_prefix('ratio_')
    .reset_index()
)

# Флаги для важных событий
flags = events_train.assign(
    has_favorite_add = events_train['event_name'] == 'favorite_add',
    has_login        = events_train['event_name'] == 'login',
    has_captcha      = events_train['event_name'] == 'captcha_shown',
).groupby('cookie_id')[['has_favorite_add', 'has_login', 'has_captcha']].any().astype(int).reset_index()

# Отношение item_view / photo_swipe
item_view   = events_train[events_train['event_name'] == 'item_view'].groupby('cookie_id').size()
photo_swipe = events_train[events_train['event_name'] == 'photo_swipe'].groupby('cookie_id').size()

ratio = (item_view / (photo_swipe + 1)).fillna(0).reset_index(name='item_view_to_photo_ratio')

# Собираем всё вместе
event_features = event_type_counts.merge(flags, on='cookie_id', how='left')
event_features = event_features.merge(ratio, on='cookie_id', how='left')

base_features = base_features.merge(event_features, on='cookie_id', how='left')

In [70]:
print(f"Всего признаков: {len(base_features.columns) - 2}")  # минус cookie_id и target

Всего признаков: 46


## **2.3. Дополнительный feature engineering**

После построения бейзлайна были выявлены слабые места полученых базовых фич, текущая задача доработать и усложнить признаки для улучшения качества модели.

### **2.3.1. Интервалы времени**

Наиболее важным признаком при классификации для модели оказались интервалы времени между событиями. Поэтому дополним данные признаки:

- `std_interval` - стандартное отклонение (вариативночть поведения)

- `iqr_interval` - межквартильный размах (устойчив к выбросам)

- `events_per_second` - средняя скорость событий

- `max_events_in_10s` - максимум событий в скользящем окне 10 секунд

- `unique_hours` - количество уникальных часов

- `peak_hour` - час с максимальной активностью

In [32]:
ev = events_train.sort_values(["cookie_id", "event_ts"]).copy()

def build_interval_features(ev: pd.DataFrame) -> pd.DataFrame:
    # dt между последовательными событиями внутри куки
    ev["delta"] = ev.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()

    g = ev.groupby("cookie_id")

    out = pd.DataFrame(index=g.size().index)

    def safe_iqr(x):
        x = x.dropna()
        if len(x) < 2:
            return np.nan
        return x.quantile(0.75) - x.quantile(0.25)

    out["std_interval"] = g["delta"].std()
    out["iqr_interval"] = g["delta"].apply(safe_iqr)

    # скорость
    span = g["event_ts"].agg(lambda x: (x.max() - x.min()).total_seconds())
    n = g.size()
    out["events_per_second"] = n / span.replace(0, np.nan)

    # максимум событий в скользящем окне 10 секунд
    def max_in_10s(times):
        times = times.sort_values().values.astype("datetime64[ns]")
        if len(times) < 2:
            return len(times)
        # два указателя: правый бежит, левый держит окно 10 сек
        left = 0
        best = 0
        for right in range(len(times)):
            while (times[right] - times[left]) / np.timedelta64(1, "s") > 10:
                left += 1
            best = max(best, right - left + 1)
        return best

    out["max_events_in_10s"] = g["event_ts"].apply(max_in_10s)

    # уникальные часы и пиковый час
    ev["hour"] = ev["event_ts"].dt.hour
    out["unique_hours"] = ev.groupby("cookie_id")["hour"].nunique()

    def peak_hour(x):
        vc = x.value_counts()          # сколько событий в каждый час
        if vc.empty:
            return -1
        return int(vc.idxmax())        # час с максимумом

    out["peak_hour"] = ev.groupby("cookie_id")["hour"].apply(peak_hour)

    out["peak_hour"] = ev.groupby("cookie_id")["hour"].apply(peak_hour)

    return out.reset_index()


interval_feats = build_interval_features(ev)
interval_feats.head()

,cookie_id,std_interval,iqr_interval,events_per_second,max_events_in_10s,unique_hours,peak_hour
0,ck_000c95f1408dcb00,3080.553503,3717.00,0.000531,1,6,19
1,ck_000e8c52636e3bec,774.305983,37.00,0.005089,3,3,7
2,ck_0010e31baa4a1fb7,3107.706813,5501.00,0.000378,2,8,23
3,ck_0010ec3874fb5378,1984.404925,80.00,0.001297,3,11,3
4,ck_001722063b94cae0,427.831357,243.25,0.003643,2,2,18


In [9]:
df_interval = interval_feats.merge(
    train[["cookie_id", "target"]], on="cookie_id", how="left"
)

feat_cols = ["std_interval", "iqr_interval", "events_per_second",
             "max_events_in_10s", "unique_hours", "peak_hour"]

print("Распределение фич по классам:")
for col in feat_cols:
    humans = df_interval.loc[df_interval["target"] == 0, col].dropna()
    bots   = df_interval.loc[df_interval["target"] == 1, col].dropna()

    print(f"\n{col}")
    print(f"  человек (n={len(humans):>5}): "
          f"median={humans.median():>8.3f}  mean={humans.mean():>8.3f}")
    print(f"  бот     (n={len(bots):>5}): "
          f"median={bots.median():>8.3f}  mean={bots.mean():>8.3f}")

Распределение фич по классам:

std_interval
  человек (n= 9574): median=1740.379  mean=1707.617
  бот     (n=  885): median=1370.304  mean=1360.315

iqr_interval
  человек (n= 9574): median=  90.000  mean= 610.307
  бот     (n=  885): median=  26.000  mean= 270.247

events_per_second
  человек (n= 9939): median=   0.002  mean=   0.010
  бот     (n=  894): median=   0.002  mean=   0.012

max_events_in_10s
  человек (n=10192): median=   2.000  mean=   1.788
  бот     (n=  899): median=   2.000  mean=   2.154

unique_hours
  человек (n=10192): median=   3.000  mean=   3.138
  бот     (n=  899): median=   3.000  mean=   3.225

peak_hour
  человек (n=10192): median=  15.000  mean=  14.023
  бот     (n=  899): median=  13.000  mean=  12.809


**Что берём в модель:**

- `std_interval` - оставляем, слабый, но сработает в комбинации.

- `iqr_interval` - **основная фича**.

- Остальные 4 (`events_per_second`, `max_events_in_10s`, `unique_hours`,`peak_hour`) — **выкидываем**, сигнала нет.

Чуть более подробнее рассмотрим признаки сгенерированные с использование процентилей, поскольку `icr_interval` - хорошо разделяет два класса:

- `p10_interval` - 10-й процентиль

- `p90_interval` - 90-й процентиль

А также рассмотрим относительные меры регулярности:

- `iqr_over_median` - IQR / медиана. Нормированная мера разброса.

- `p90_over_p10` - отношение хвостов.

- `cv_interval` - коэффициент вариации (std/mean)

- `std_over_median` - std / медиана

Доли интервалов в диапазонах:

- `ratio_gt_60s` - доля интервалов > 60 сек. Боты не делают длинных пауз.

- `ratio_gt_300s` - доля интервалов > 5 мин

In [47]:
def build_interval_features(ev: pd.DataFrame) -> pd.DataFrame:
    # сортируем события по куке и времени.
    ev = ev.sort_values(["cookie_id", "event_ts"]).copy()

    # считаем delta = секунды между текущим и предыдущим событием
    ev["delta"] = (
        ev.groupby("cookie_id")["event_ts"]
          .diff()
          .dt.total_seconds()
    )

    # Дальше работаем через groupby-объект
    g = ev.groupby("cookie_id")

    # Результат
    out = pd.DataFrame(index=g.size().index)

    # Базовые интервалы
    out["median_interval"] = g["delta"].median()
    out["mean_interval"] = g["delta"].mean()
    out["std_interval"] = g["delta"].std()
    out["min_interval"] = g["delta"].min()
    out["max_interval"] = g["delta"].max()

    # Перцентили 
    out["p10_interval"] = g["delta"].quantile(0.10)
    out["p90_interval"] = g["delta"].quantile(0.90)

    # IQR считаем вручную
    def iqr(x: pd.Series) -> float:
        x = x.dropna()
        if len(x) < 2:
            return np.nan
        return x.quantile(0.75) - x.quantile(0.25)

    out["iqr_interval"] = g["delta"].apply(iqr)

    # Относительные меры регулярности
    out["iqr_over_median"] = out["iqr_interval"] / out["median_interval"].replace(0, np.nan)
    out["std_over_median"] = out["std_interval"] / out["median_interval"].replace(0, np.nan)
    out["p90_over_p10"] = out["p90_interval"] / out["p10_interval"].replace(0, np.nan)
    out["cv_interval"] = out["std_interval"] / out["mean_interval"].replace(0, np.nan)

    # Доли интервалов в диапазонах
    def ratio_gt(x: pd.Series, threshold: float) -> float:
        """Доля интервалов больше threshold"""
        x = x.dropna()
        if len(x) == 0:
            return np.nan
        return (x > threshold).mean()

    out["ratio_gt_60s"]  = g["delta"].apply(lambda x: ratio_gt(x, 60))
    out["ratio_gt_300s"] = g["delta"].apply(lambda x: ratio_gt(x, 300))

    return out.reset_index()

interval_feats_v2 = build_interval_features(events_train)
interval_feats_v2.head()

,cookie_id,median_interval,mean_interval,std_interval,min_interval,max_interval,p10_interval,p90_interval,iqr_interval,iqr_over_median,std_over_median,p90_over_p10,cv_interval,ratio_gt_60s,ratio_gt_300s
0,ck_000c95f1408dcb00,58.0,2007.800000,3080.553503,17.0,7733.0,18.8,7316.8,3717.00,64.086207,53.112991,389.191489,1.534293,0.466667,0.333333
1,ck_000e8c52636e3bec,23.0,202.454545,774.305983,0.0,4257.0,2.4,64.0,37.00,1.608696,33.665478,26.666667,3.824592,0.121212,0.060606
2,ck_0010e31baa4a1fb7,334.0,2825.066667,3107.706813,10.0,8243.0,24.4,6166.2,5501.00,16.470060,9.304511,252.713115,1.100047,0.800000,0.533333
3,ck_0010ec3874fb5378,32.0,781.698630,1984.404925,0.0,8567.0,5.2,2797.6,80.00,2.500000,62.012654,538.000000,2.538581,0.342466,0.164384
4,ck_001722063b94cae0,175.0,294.142857,427.831357,2.0,1447.0,18.3,838.1,243.25,1.390000,2.444751,45.797814,1.454502,0.642857,0.214286


### **2.3.3. UA-признаки**

Поскольку ua-признаки плохо себя показали в модели, стоит их пересмотреть. Попробуем следующий набор фич:

- `ratio_is_bot_ua` - заменяет any, ловит «смешанные» куки, где бот-UA только часть событий.

- `is_pure_bot_ua` - кука, где все события с бот-UA - почти 100% бот

- `ratio_is_mobile_app` - okhttp = мобильное приложение, а не бот, модель отделит app от web

- `ua_length_median` - аномально короткий/длинный UA.

In [21]:
def _ua_features(events: pd.DataFrame) -> pd.DataFrame:
    ua = events[["cookie_id", "user_agent"]].copy()
    ua["ua"] = ua["user_agent"].fillna("").str.lower()

    # явные библиотеки парсинга
    ua["is_bot_ua"] = ua["ua"].str.contains(
        r"headlesschrome|scrapy|python-|urllib|curl/|wget|go-http|node-fetch",
        regex=True,
    )
    # okhttp - мобильное приложение
    ua["is_mobile_app"] = ua["ua"].str.contains("okhttp", regex=False)

    g = ua.groupby("cookie_id")
    out = pd.DataFrame(index=g.size().index)

    # доля событий с бот-UA
    out["ratio_is_bot_ua"] = g["is_bot_ua"].mean()

    # чистая кука = все события с бот-UA
    out["is_pure_bot_ua"] = (out["ratio_is_bot_ua"] == 1.0).astype(int)

    # флаг мобильного приложения (отделяет Android/iOS-app от web)
    out["ratio_is_mobile_app"] = g["is_mobile_app"].mean()

    # длина UA как «аномалия короткого клиента»
    out["ua_length_median"] = g.apply(
        lambda x: x["ua"].str.len().median()
    )

    return out.reset_index()

### **2.3.4. Нормализованные счетчики**

Сейчас используем абсолютные счетчики:

- `event_count` - сколько всего событий
- `unique_items` - сколько уникальных объявлений
- `unique_categories` - сколько уникальных категорий
- `unique_locations` - сколько уникальных локаций
- `unique_search_queries` - сколько уникальных запросов

Но они смешаны с длиной сессии. Кука, которая была на сайте 5 минут, и кука, которая была 5 часов, - по этим фичам выглядят по-разному просто потому что они разной длины. Введем:

- `items_per_event`

- `categories_per_event`

- `locations_per_event`

- `queries_per_event`

- `event_types_per_event`

In [28]:
def _normalized_count_features(events: pd.DataFrame) -> pd.DataFrame:
    """
    Нормализованные счётчики: делим 'уникальное' на длину сессии.
    """
    g = events.groupby("cookie_id")

    out = pd.DataFrame(index=g.size().index)

    out["event_count"] = g.size()

    out["unique_items"]      = g["item_id"].nunique()
    out["unique_categories"] = g["item_category"].nunique()
    out["unique_locations"]  = g["item_location"].nunique()
    out["unique_event_types"] = g["event_name"].nunique()

    q = events[events["search_query"].fillna("").str.strip() != ""]
    out["unique_search_queries"] = (
        q.groupby("cookie_id")["search_query"].nunique()
        .reindex(out.index, fill_value=0)
    )

    # нормализация
    n = out["event_count"].clip(lower=1)   # защита от 0

    out["items_per_event"] = out["unique_items"] / n
    out["categories_per_event"] = out["unique_categories"] / n
    out["locations_per_event"] = out["unique_locations"]  / n
    out["queries_per_event"] = out["unique_search_queries"] / n
    out["event_types_per_event"] = out["unique_event_types"]  / n

    return out.reset_index()


norm_feats = _normalized_count_features(events_train)
norm_feats.head()

,cookie_id,event_count,unique_items,unique_categories,unique_locations,unique_event_types,unique_search_queries,items_per_event,categories_per_event,locations_per_event,queries_per_event,event_types_per_event
0,ck_000c95f1408dcb00,16,9,2,8,4,3,0.562500,0.125000,0.500000,0.187500,0.250000
1,ck_000e8c52636e3bec,34,20,4,12,7,8,0.588235,0.117647,0.352941,0.235294,0.205882
2,ck_0010e31baa4a1fb7,16,8,3,4,6,4,0.500000,0.187500,0.250000,0.250000,0.375000
3,ck_0010ec3874fb5378,74,31,2,19,6,8,0.418919,0.027027,0.256757,0.108108,0.081081
4,ck_001722063b94cae0,15,8,3,9,4,4,0.533333,0.200000,0.600000,0.266667,0.266667


### **2.3.5. Признаки на основе n-грамм**

Отличительной чертой ботов от людей является то, что они действуют циклично, используя одну и ту же последовательность действий. Человек же, наоборот, действует более разнообразно. Попробуем это уловить используя n-граммы.

Признаки будем строить по отсортированной последовательности `event_name`:

- `bigrams_per_transition` = `unique_bigrams` / `n_bigrams`

- `trigrams_per_transition` = `unique_trigrams` / `n_trigrams`

- `unique_trigrams` - число уникальных триграмм

In [ ]:
def build_transition_features(ev: pd.DataFrame) -> pd.DataFrame:
    ev = ev.sort_values(["cookie_id", "event_ts"])

    rows = []
    for cookie_id, sub in ev.groupby("cookie_id", sort=False):
        seq = sub["event_name"].tolist()    # получаем список событий в отсортированном виде
        n_events = len(seq)
        n_big = n_events - 1    # кол-во биграмм
        n_tri = n_events - 2    # кол-во триграмм

        row = {"cookie_id": cookie_id}

        # Короткая сессия: переходов нет, оставляем NaN
        if n_events < 3:
            row.update({
                "bigrams_per_transition": np.nan,
                "unique_trigrams": 0,
                "trigrams_per_transition": np.nan,
            })
            rows.append(row)
            continue

        # Получаем биграммы и триграммы
        bigrams  = list(zip(seq[:-1], seq[1:]))
        trigrams = list(zip(seq[:-2], seq[1:-1], seq[2:]))

        n_unique_bigrams  = len(set(bigrams))
        n_unique_trigrams = len(set(trigrams))
        
        row["bigrams_per_transition"]  = n_unique_bigrams / n_big
        row["unique_trigrams"]         = n_unique_trigrams
        row["trigrams_per_transition"] = n_unique_trigrams / n_tri

        rows.append(row)

    return pd.DataFrame(rows)

### **2.3.6. Признаки на основе данных координат курсора**

Также стоит капнуть чуть глубже в координаты курсора, и сделать больше признаков.

Фичи будем считать только по событиям, где `pointer_x/y` заполнены.

- `ptr_mean_step` - среднее расстояние между соседними точками

- `ptr_std_x/y` - стандартное отклонение X/Y

- `ptr_range_x/y` - размах X/Y: `max - min`

In [ ]:
def build_pointer_features(ev: pd.DataFrame) -> pd.DataFrame:
    ev = ev.sort_values(["cookie_id", "event_ts"])

    rows = []
    for cookie_id, sub in ev.groupby("cookie_id", sort=False):
        mask = sub["pointer_x"].notna() & sub["pointer_y"].notna()

        row = {"cookie_id": cookie_id}

        if mask.sum() < 2:
            row.update({k: np.nan for k in [
                "ptr_std_x", "ptr_std_y", "ptr_range_x", "ptr_range_y",
                "ptr_mean_step"
            ]})
            rows.append(row)
            continue

        p = sub.loc[mask, ["pointer_x", "pointer_y"]]
        x = p["pointer_x"].to_numpy()
        y = p["pointer_y"].to_numpy()

        # Разброс координат
        row["ptr_std_x"]   = float(np.std(x))
        row["ptr_std_y"]   = float(np.std(y))
        row["ptr_range_x"] = float(x.max() - x.min())
        row["ptr_range_y"] = float(y.max() - y.min())

        # Шаги между последовательными точками
        steps = np.hypot(np.diff(x), np.diff(y))    # np.hypot - ищет расстояние между точками по теореме пифагора
        row["ptr_mean_step"] = float(steps.mean())

        rows.append(row)

    return pd.DataFrame(rows)